### Загрузка датасета

In [12]:
from datasets import load_from_disk
import numpy as np

# ds = load_from_disk("/kaggle/input/datasets/typaya/manipulative-ru-patterns/ru_token_cls_dataset")
ds = load_from_disk("ru_token_cls_dataset")
ds.set_format(type=None)

print(ds)
print(f"\nTrain: {len(ds['train'])}")
print(f"Val: {len(ds['validation'])}")
print(f"Test: {len(ds['test'])}")

example = ds["train"][0]["labels"]
n_classes = np.array(example).shape[1]

print(f"\nЧисло классов (n_classes): {n_classes}")

id2label = {i: f"LABEL_{i}" for i in range(n_classes)}
label2id = {v: k for k, v in id2label.items()}

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 1660
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 293
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 489
    })
})

Train: 1660
Val: 293
Test: 489

Число классов (n_classes): 10


In [ ]:
# !pip install seqeval -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### Обучение модели

In [ ]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForTokenClassification, EarlyStoppingCallback
import numpy as np
import torch
from torch import nn
from sklearn.metrics import precision_recall_fscore_support


MODEL_NAME = "DeepPavlov/rubert-base-cased"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id,
)

class MultiLabelTokenCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=None):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, features):
        labels = [f["labels"] for f in features]
        features = [{k: v for k, v in f.items() if k != "labels"} for f in features]

        batch = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
            pad_to_multiple_of=self.pad_to_multiple_of
        )

        max_len = batch["input_ids"].shape[1]
        n_classes = len(labels[0][0])

        padded_labels = []
        for lab in labels:
            lab = torch.as_tensor(lab, dtype=torch.float32)
            pad_len = max_len - lab.shape[0]
            if pad_len > 0:
                pad = torch.full((pad_len, n_classes), -100.0)
                lab = torch.cat([lab, pad], dim=0)
            padded_labels.append(lab)

        batch["labels"] = torch.stack(padded_labels)
        return batch

data_collator = MultiLabelTokenCollator(tokenizer)

counts = np.zeros(len(label2id))
total_tokens = 0

for ex in ds["train"]:
    lab = np.array(ex["labels"])
    mask = ~(lab == -100).all(axis=1)
    lab = lab[mask]
    total_tokens += lab.shape[0]
    counts += lab.sum(axis=0)

pos = np.maximum(counts, 1)
neg = np.maximum(total_tokens - pos, 1)
pos_weight_np = np.clip(neg / pos, 1.0, 50.0) 
pos_weight = torch.tensor(pos_weight_np, dtype=torch.float32)

print("Веса классов (pos_weight):", pos_weight)

class FocalLoss(nn.Module):
    def __init__(self, pos_weight, gamma=2.0, reduction="mean"):
        super().__init__()
        self.pos_weight = pos_weight
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none", pos_weight=self.pos_weight.to(logits.device)
        )
        probs = torch.sigmoid(logits)
        
        pt = probs * targets + (1 - probs) * (1 - targets)
        
        loss = ((1 - pt) ** self.gamma) * bce

        if self.reduction == "mean":
            return loss.mean()
        return loss.sum()

class MultiLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        mask = (labels != -100).any(dim=-1)
        logits = logits[mask]
        labels = labels[mask]

        loss_fct = FocalLoss(pos_weight=pos_weight, gamma=2.0)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss
        
def compute_metrics(eval_pred, threshold=0.3):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    mask = labels[:, :, 0] != -100
    probs, labels = probs[mask], labels[mask]

    preds = (probs > threshold).astype(int)
    
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)
    
    return {"precision_macro": p_macro, "recall_macro": r_macro, "f1_macro": f1_macro,
            "precision_micro": p_micro, "recall_micro": r_micro, "f1_micro": f1_micro}


training_args = TrainingArguments(
    output_dir="./rubert_multilabel",
    num_train_epochs=15,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=3e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
)


trainer = MultiLabelTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


trainer.train()
trainer.save_model("./manipulation_detector_model")
print("\nМодель сохранена в ./manipulation_detector_model")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/714M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import shutil
from pathlib import Path

MODEL_DIR = Path("./manipulation_detector_model")
ZIP_PATH = Path("./manipulation_detector_model.zip")

if MODEL_DIR.exists():
    shutil.make_archive(str(ZIP_PATH.stem), 'zip', MODEL_DIR)
    print(f"Архив успешно создан: {ZIP_PATH.absolute()}")
else:
    print(f"Папка {MODEL_DIR} не найдена. Проверь путь в trainer.save_model()")

Архив успешно создан: /kaggle/working/manipulation_detector_model.zip


### Тестирование модели

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

label_ru = {
    "emotional_triggering_language": "эмоционально нагруженная лексика",
    "fear_uncertainty_pressure": "страх / давление неопределённостью",
    "doubt_uncertainty_injection": "вброс сомнений",
    "cognitive_closure_cliches": "клише / когнитивное закрытие",
    "social_proof_pressure": "социальное давление / большинство",
    "gain_loss_exaggeration": "преувеличение выигрыша/потерь",
    "causal_fact_distortion": "искажение причин/фактов",
    "authority_claim_pressure": "апелляция к авторитету",
    "topic_shift_misrepresentation": "подмена темы / искажение",
    "directive_action_pressure": "призыв к действию / давление",
}

id2label = {i: lab for i, lab in enumerate(label_ru)}
label2id = {v: k for k, v in id2label.items()}

MODEL_PATH = "./manipulation_detector_model_first"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)
model.eval()

def predict_and_explain_multilabel(text, threshold=0.5):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        logits = model(**inputs).logits[0]

    probs = torch.sigmoid(logits)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    token_labels = []
    token_probs = []

    for i, tok in enumerate(tokens):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            token_labels.append([])
            token_probs.append([])
            continue

        active = (probs[i] >= threshold).nonzero(as_tuple=True)[0].tolist()
        labels = [id2label[j] for j in active]
        scores = [probs[i, j].item() for j in active]

        token_labels.append(labels)
        token_probs.append(scores)

    words, word_labels, word_probs = [], [], []
    cur_word, cur_labels, cur_probs = "", [], []

    for tok, labs, probs_ in zip(tokens, token_labels, token_probs):
        if tok in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        if tok.startswith("##"):
            cur_word += tok[2:]
        else:
            if cur_word:
                words.append(cur_word)
                word_labels.append(cur_labels)
                word_probs.append(cur_probs)
            cur_word = tok
            cur_labels = labs
            cur_probs = probs_

    if cur_word:
        words.append(cur_word)
        word_labels.append(cur_labels)
        word_probs.append(cur_probs)

    spans = []
    for label_name in label_ru:
        i = 0
        while i < len(words):
            if label_name in word_labels[i]:
                span_text = words[i]
                max_prob = word_probs[i][word_labels[i].index(label_name)]
                i += 1
                while i < len(words) and label_name in word_labels[i]:
                    span_text += " " + words[i]
                    max_prob = max(max_prob, word_probs[i][word_labels[i].index(label_name)])
                    i += 1
                spans.append({
                    "text": span_text,
                    "pattern_name": label_name,
                    "confidence": max_prob
                })
            else:
                i += 1

    result = f"""

ТЕКСТ:
{text}

РЕЗУЛЬТАТ АНАЛИЗА:
"""

    if spans:
        result += "МАНИПУЛЯЦИЯ ОБНАРУЖЕНА\n\n"
        patterns = {}
        for s in spans:
            patterns.setdefault(s["pattern_name"], []).append(s)

        for idx, (pname, plist) in enumerate(patterns.items(), 1):
            result += f"{idx}️. Манипуляция {pname} ({label_ru.get(pname, '-')})\n"
            result += f"      Примеры:\n"
            for s in plist:
                result += f"        - \"{s['text']}\" (conf: {s['confidence']*100:.2f}%)\n"
            result += "\n"
    else:
        result += "МАНИПУЛЯЦИЯ НЕ ОБНАРУЖЕНА\n"

    result += "\n"
    return result, spans


test_texts = [
    "Все нормальные люди голосуют за нас. Если ты не согласен, ты явно глуп.",
    "Все нормальные люди знают что это правда.",
    "Пожалуйста, помогите мне понять эту проблему.",

    "Пока вы спокойно пьёте кофе, элиты уже давно продают вашу безопасность за чужие интересы. "
    "Миллионы граждан уже вышли на улицы, потому что понимают: тянуть больше нельзя. "
    "Требуйте немедленных реформ, пока не стало слишком поздно. Это единственный шанс спасти страну от катастрофы.",

    "Ведущие эксперты финансового рынка единогласно предупреждают: традиционные вклады давно утратили смысл "
    "в условиях скрытой инфляции. Те, кто ещё не перешёл на защищённые цифровые активы, просто упускают "
    "последнее окно возможностей. Очевидно, что здравомыслящий человек не станет ждать, когда его сбережения "
    "превратятся в пыль. Действуйте сейчас, потому что завтра этот путь будет закрыт.",

    "Вот вам очередной пример того, как оппозиция пытается выставить правительство агрессором, полностью "
    "игнорируя тот факт, что именно их законопроекты спровоцировали рост цен на лекарства. "
    "Вместо того чтобы решать реальные проблемы семей, они продолжают раздувать искусственные скандалы. "
    "Но разве мы не обязаны ставить во главу угла справедливость, поддержку материнства и защиту наших детей? "
    "Давайте вернёмся к обсуждению настоящей социальной политики.",

    "Мы наблюдаем невероятный подъём доверия к нашей экосистеме: тысячи пользователей уже отмечают, как "
    "качественно изменилась их повседневная жизнь. Критики, конечно, любят цепляться за единичные баги, "
    "но вспомните, сколько раз сами IT-гиганты выпускали сырые обновления. Прогресс не терпит сомнений. "
    "Присоединяйтесь к сообществу тех, кто уже строит будущее вместе с нами.",

    "Совпадение ли, что сразу после принятия нового закона о цифровом профиле участились утечки данных "
    "и начали блокировать независимые каналы? Специалисты давно бьют тревогу: это не случайность, "
    "а тщательно спланированная схема контроля. Хватит кормить систему своим молчанием. Отключите слежку, "
    "пока вас окончательно не превратили в прозрачного потребителя.",

    "Сегодня в 14:00 в конференц-зале состоится плановое собрание отдела аналитики. "
    "Повестка: обсуждение квартальных метрик и распределение задач на следующий месяц. "
    "Просьба подготовить отчёты заранее."
]

for text in test_texts:
    result, _ = predict_and_explain_multilabel(text)
    print(result)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]



ТЕКСТ:
Все нормальные люди голосуют за нас. Если ты не согласен, ты явно глуп.

РЕЗУЛЬТАТ АНАЛИЗА:
МАНИПУЛЯЦИЯ ОБНАРУЖЕНА

1️. Манипуляция gain_loss_exaggeration (преувеличение выигрыша/потерь)
      Примеры:
        - "Все нормальные люди голосуют за нас . Если ты не согласен , ты явно глуп ." (conf: 79.53%)





ТЕКСТ:
Все нормальные люди знают что это правда.

РЕЗУЛЬТАТ АНАЛИЗА:
МАНИПУЛЯЦИЯ ОБНАРУЖЕНА

1️. Манипуляция gain_loss_exaggeration (преувеличение выигрыша/потерь)
      Примеры:
        - "Все нормальные люди знают что это правда ." (conf: 75.37%)





ТЕКСТ:
Пожалуйста, помогите мне понять эту проблему.

РЕЗУЛЬТАТ АНАЛИЗА:
МАНИПУЛЯЦИЯ НЕ ОБНАРУЖЕНА




ТЕКСТ:
Пока вы спокойно пьёте кофе, элиты уже давно продают вашу безопасность за чужие интересы. Миллионы граждан уже вышли на улицы, потому что понимают: тянуть больше нельзя. Требуйте немедленных реформ, пока не стало слишком поздно. Это единственный шанс спасти страну от катастрофы.

РЕЗУЛЬТАТ АНАЛИЗА:
МАНИПУЛЯЦИЯ ОБНАРУ